# Configure Forcings (`case.configure_forcings`)

`configure_forcings` declares every forcing your case needs, initial conditions, open
boundary conditions, tides, biogeochemistry, chlorophyll, runoff, and is always
required before [Process Forcings](process_forcings.ipynb). What arguments you owe it
depends on your `compset` (see [Case Setup, Section 2](case_setup.ipynb#case-compset-requires)
for how to discover the required list before you get here).

This notebook covers:
- [Section 1](#forcings-date-range): the one always-required argument, `date_range`
- [Section 2](#forcings-data-products): switching data products/access functions (not GLORYS)
- [Section 3](#forcings-tides): tides
- [Section 4](#forcings-chlorophyll): chlorophyll
- [Section 5](#forcings-runoff): runoff (GLOFAS/JRA via compset, or a custom product)
- [Section 6](#forcings-bgc): biogeochemistry (MARBL), the kwargs half of [Case Setup, Section 4](case_setup.ipynb#case-marbl)
- [Section 7](#forcings-cice): sea ice (CICE6), the kwargs half of [Case Setup, Section 3](case_setup.ipynb#case-cice)
- [Section 8](#forcings-ww3): waves (WW3), the kwargs half of [Case Setup, Section 5](case_setup.ipynb#case-ww3)

📖 [CrocoDash configure_forcings docs](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/3a_configure_forcings.html) · [Datasets](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/datasets.html)

(forcings-date-range)=
## Section 1: The Required Argument, `date_range`

We need to cut out our ocean forcing. The package expects an initial condition and one
time-dependent segment per non-land boundary. Naming convention is `"east_unprocessed"`
for segments and `"ic_unprocessed"` for the initial condition.

By default, `configure_forcings` forces with the Copernicus Marine "Glorys" reanalysis
dataset. `date_range` is always required. It drives the initial and boundary condition
extraction that every case needs, and optional configurators (like tides) derive their
reference dates from it.

In [ ]:
case.configure_forcings(
    date_range = ["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    boundaries=["south","east","west"],
    function_name="get_glorys_data_from_rda"
)

(forcings-data-products)=
## Section 2: Switching Data Products

This module can be used with `case.configure_forcings` to find different ways to access
raw data from data sources. Just supply a data product name and function name:

In [ ]:
case.configure_forcings(date_range = ["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
product_name = "GLORYS",
function_name = "get_glorys_data_from_rda")

Available products and functions can be found in the
[documentation](https://crocodile-cesm.github.io/CrocoDash/latest/for_users/datasets.html)
and in the `raw_data_access` helper functions shown below.

In [ ]:
from CrocoDash.raw_data_access.registry import ProductRegistry

In [ ]:
ProductRegistry.load() # Static object, no need to instantiate it (It's a registry)
ProductRegistry.list_products() # List all registered products

In [ ]:
ProductRegistry.list_access_methods("GLORYS")

In [ ]:
ProductRegistry.get_product("GLORYS")

In [ ]:
# For example, with what we printed above, we can configure the case to use the GLORYS product and a function from that product as follows:

case.configure_forcings(date_range = ["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
                       product_name = "GLORYS",
                       function_name = "get_glorys_data_from_cds_api")

### Accessing Non-Forcing Raw Data

Apart from accessing forcing products through `case.configure_forcings()`, we can access
products like GEBCO, SEAWIFS, GLOFAS, etc. by importing the raw data access module
directly.

In [ ]:
# Import the specific module (which can be found by looking at the API Documentation: https://crocodile-cesm.github.io/CrocoDash/latest/api-docs/CrocoDash.raw_data_access.datasets.html)
from CrocoDash.raw_data_access.datasets import glofas as gl

# Then call the function
gl.GLOFAS.get_processed_global_glofas_script_for_cli(output_folder="sample", output_filename="glofas_processed_data.nc")


# OR
ProductRegistry.get_access_function("GLOFAS","get_processed_global_glofas_script_for_cli")(output_folder="sample", output_filename="glofas_processed_data.nc")

(forcings-tides)=
## Section 3: Tides

MOM6 can take tides data as a boundary condition in the regional domain. Many tides
parameters are impacted by this and can be seen in `case.process_forcings`.

In our workflow, we take data from the TPXO tidal model and regrid onto our grid. TPXO
model data can be requested off of the TPXO website or is available on Derecho.

The file paths of the tidal files can be passed into `configure_forcings` as shown below
with all wanted tidal constituents, like M2. There are three parameters in
`configure_forcings` and many parameters adjusted in MOM6.

In [ ]:
case.configure_forcings(
    date_range = ["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    tidal_constituents = ['M2'],
    tpxo_elevation_filepath = "<TPXO_H>",
    tpxo_velocity_filepath = "<TPXO_U>"
)

```{note}
If you're working on **Derecho** or **Casper**, you can use the following paths:

```{code-block} python
TPXO Elevation: "/glade/campaign/cesm/cesmdata/inputdata/ocn/mom/croc/crocogallerydata/tpxo/h_tpxo9.v1.nc"
TPXO Velocity: "/glade/campaign/cesm/cesmdata/inputdata/ocn/mom/croc/crocogallerydata/tpxo/u_tpxo9.v1.nc"
```
```

(forcings-chlorophyll)=
## Section 4: Chlorophyll

MOM6 can take chlorophyll data as a file in the regional domain. It impacts shortwave
penetration. MOM6 parameters that are impacted by it are `CHL_FROM_FILE`, `CHL_FILE`,
`VAR_PEN_SW`, and `PEN_SW_NBANDS`. In our workflow, we take raw data from SeaWIFS,
process it globally, and subset to our regional domain.

Chlorophyll can be added into a CrocoDash case using functions from
`regional_mom6.chl`, called from `configure_forcings`. There is one parameter in
`configure_forcings` and four parameters in MOM6.

```{caution}
This method does not do a great job of resolving chlorophyll in estuaries and similar
features. If possible, the generated chlorophyll file should be replaced with a better
product (which can be done by replacing the file in `CHL_FILE`).
```

### CrocoDash Parameters

In `case.configure_forcings()`, the argument `chl_processed_filepath` takes in a
processed global chlorophyll file. The global processed chlorophyll file is hosted on
the CESM inputdata svn server under `ocn/mom/croc/chl/data` and can be accessed through
the CrocoDash `raw_data_access` module like below:

In [ ]:
from CrocoDash.raw_data_access.datasets import seawifs as sw
sw.SeaWIFS.get_processed_global_seawifs_script_for_cli(
    output_folder="<insert_dir>",
    output_filename="get_seawifs_data.sh"
)

The file path of the global file (after running the script from the code block) can be passed into `configure_forcings` as shown in this demo:

In [ ]:
case.configure_forcings(
    date_range = ["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    chl_processed_filepath = "<CHL>",
)

```{note}
If you're working on **Derecho** or **Casper**, you can use the following path:

```{code-block} python
chl_processed_filepath = "/glade/campaign/cesm/cesmdata/inputdata/ocn/mom/croc/chl/data/SeaWIFS.L3m.MC.CHL.chlor_a.0.25deg.nc"
```
```

(forcings-runoff)=
## Section 5: Runoff

River runoff adds fresh-water discharge at the ocean surface. CrocoDash supports two
routes:

- **[Section 5.1](#forcings-runoff-compset)**: use GLOFAS or JRA runoff that ships with CESM (recommended starting point). The compset choice for this lives in [Case Setup](case_setup.ipynb) (see the compset quick-reference table).
- **[Section 5.2](#forcings-runoff-custom)**: plug in a custom runoff dataset (advanced).

(forcings-runoff-compset)=
### Section 5.1: GLOFAS or JRA via Compset

Both GLOFAS and JRA runoff products ship with CESM. To activate either, build your
`Case` with a compset alias that includes runoff (e.g. `CR_JRA_GLOFAS`, see
[Case Setup](case_setup.ipynb)), then pass the ESMF mesh file for your chosen dataset to
`configure_forcings` so CrocoDash can compute the remapping weights.

In [ ]:
case.configure_forcings(
    date_range=["2000-01-01 00:00:00", "2000-02-01 00:00:00"],
    rof_esmf_mesh_filepath="<GLOFAS_MESH>",
)

```{note}
On **Derecho/Casper** the mesh files are already available:

```{code-block} python
# GLOFAS
rof_esmf_mesh_filepath = "/glade/campaign/cesm/cesmdata/inputdata/ocn/mom/croc/rof/glofas/dis24/GLOFAS_esmf_mesh_v4.nc"
# JRA
rof_esmf_mesh_filepath = "/glade/campaign/cesm/cesmdata/inputdata/lnd/dlnd7/JRA55/JRA.v1.4.runoff.1958_ESMFmesh_cdf5_20201020.nc"
```
```

(forcings-runoff-custom)=
### Section 5.2: Custom Runoff Product (Advanced)

To use a custom runoff dataset you need three things: an ESMF mesh file, a stream
definition file, and the grid dimensions of your raw data.

#### Step 1: Create an ESMF Mesh File

Generate a mesh file from a `Grid` that matches your raw data's grid:

In [ ]:
from CrocoDash.grid import Grid
from CrocoDash.topo import Topo

grid = Grid(
    lenx=360,
    leny=150,
    cyclic_x=True,
    ystart=-60,
    resolution=0.10,
    name="GLOFAS",
)
topo = Topo(grid, min_depth=0)
topo.set_flat(10)
topo.write_esmf_mesh("<path>")

#### Step 2: Stream Definition File

In your case directory, create `drof.streams.xml` with the content below, replacing all
`<PLACEHOLDER>` values with your actual paths and variable names:

```xml
<?xml version="1.0"?>
<file id="stream" version="2.0">
  <stream_info name="rof.<PRODUCT_NAME>">
    <taxmode>cycle</taxmode>
    <tintalgo>upper</tintalgo>
    <readmode>single</readmode>
    <mapalgo>bilinear</mapalgo>
    <dtlimit>3.0</dtlimit>
    <year_first>START_YEAR</year_first>
    <year_last>END_YEAR</year_last>
    <year_align>START_YEAR</year_align>
    <vectors>null</vectors>
    <meshfile>PATH_TO_ESMF_MESH_FILE</meshfile>
    <lev_dimname>null</lev_dimname>
    <datafiles>
      <file>PATH_TO_RAW_DATA_NETCDF3_64BIT_OFFSET</file>
    </datafiles>
    <datavars>
      <var>NETCDF_VARIABLE_NAME Forr_rofl</var>
    </datavars>
    <offset>0</offset>
  </stream_info>
</file>
```

Also register your product in `components/cdeps/drof/namelist_definition_drof.xml` and
`config_component`. See the CESM docs for details.

#### Step 3: Set Grid Dimensions

Run these `xmlchange` commands in your case directory:

```bash
./xmlchange ROF_NY=1500
./xmlchange ROF_NX=3600
./xmlchange ROF_DOMAIN_MESH=<MESH_PATH>
```

Then pass your mesh file to `configure_forcings` as shown in Section 5.1 so CrocoDash
generates the ocean remapping weights.

(forcings-bgc)=
## Section 6: Biogeochemistry (MARBL)

The compset choice for MARBL is covered in
[Case Setup, Section 4](case_setup.ipynb#case-marbl). Once
that `Case` exists, here's the `configure_forcings` half:

In [ ]:
case.configure_forcings(
    date_range=["2000-01-01 00:00:00", "2000-02-01 00:00:00"],
    product_name="cesm_pop_output",
    function_name="get_cesm_single_variable_data",
    marbl_ic_filepath="",      # path to MARBL global IC file
    # For river nutrients (requires GLOFAS runoff also enabled):
    # rof_esmf_mesh_filepath="",
    # global_river_nutrients_filepath="",
)

```{note}
On **Derecho/Casper** the `cesm_pop_output` product's `get_cesm_single_variable_data`
function already defaults `dataset_path` to a CESM-HR run with the right variables:

```{code-block} python
# dataset_path defaults to:
# /glade/campaign/collections/cmip/CMIP6/CESM-HR/FOSI_BGC/HR/g.e22.TL319_t13.G1850ECOIAF_JRA_HR.4p2z.001/ocn/proc/tseries/month_1
marbl_ic_filepath = "/glade/campaign/collections/gdex/data/d651077/cesmdata/inputdata/ocn/mom/tx0.66v1/ecosys_jan_IC_omip_latlon_1x1_180W_c231221.nc"
# River nutrients (optional):
rof_esmf_mesh_filepath = "/glade/campaign/cesm/cesmdata/inputdata/ocn/mom/croc/rof/glofas/dis24/GLOFAS_esmf_mesh_v4.nc"
global_river_nutrients_filepath = "/glade/campaign/cesm/cesmdata/inputdata/ocn/mom/croc/rof/river_nutrients/river_nutrients.GNEWS_GNM.glofas.20250916.64bit.nc"
```
```

(forcings-cice)=
## Section 7: Sea Ice (CICE6)

The compset choice for CICE is covered in
[Case Setup, Section 3](case_setup.ipynb#case-cice). Here's the `configure_forcings` half.

CICE forcing is a single **restoring file**: your regional domain plus a one-T-cell
halo on every side (CICE's own ghost-cell width, so it isn't configurable), regridded from a CICE-shaped source onto every point
of that expanded grid. It carries no `time` dimension, it's one static snapshot, so unlike
MOM6's OBC there's no date chunking.


In [ ]:
case.configure_forcings(
    date_range=["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    # ... the usual MOM6 IC/OBC arguments from Sections 1-2 ...
    restore_ice=True,
    cice_product_name="cice_restart",
    cice_function_name="get_cice_restart_subset",
    cice_function_args={
        "restart_path": "<CICE_RESTART>",   # a global CICE restart, *.cice.r.*.nc
        "grid_path": "<TX2_3V3_GRID>",      # the restart's companion grid file
    },
)

### The arguments

| Argument | Default | What it's for |
|---|---|---|
| `restore_ice` | `False` | Turn on restoring of the boundary ghost cells toward the generated file |
| `cice_product_name` | `None` | Which CICE forcing product to source the restoring target from |
| `cice_function_name` | `None` | Which access function on that product to call |
| `cice_function_args` | `{}` | Extra kwargs the access function needs (`restart_path`/`grid_path` for `cice_restart`; none for `reference_ice`) |

Restoring takes **all three** of `restore_ice=True`, `cice_product_name` and
`cice_function_name`; passing only some of them is an error. Pass none of them and CICE
runs with zero-gradient boundaries and no restoring, which is a valid configuration too.
The product arguments mirror the `product_name`/`function_name` pattern Sections 1–2 use
for MOM6's IC and OBC, and they're validated: passing a MOM6 forcing product as
`cice_product_name` is an error, it has to be a registered CICE forcing product.

### The products

| Product | Access function | Needs |
|---|---|---|
| `cice_restart` | `get_cice_restart_subset` | A real global CICE restart plus its companion grid file, index-subset to your domain's bounding box. On Derecho the tx2_3v3 grid lives at `/glade/campaign/cesm/community/omwg/grids/tx2_3v3_grid.nc` |
| `reference_ice` | `get_reference_ice_data` | Nothing. Synthetic ice concentration/volume/surface temperature and a small drift velocity, on a mesh it generates itself. Good for wiring tests and demos |

### What it writes

`user_nl_cice` gets the regional-domain settings:

| `user_nl_cice` | Set to |
|---|---|
| `ns_boundary_type` / `ew_boundary_type` | `'zero_gradient'` |
| `advection` | `'upwind'` |
| `restart_ext` | `.true.` |
| `restore_ice` | `.true.` when restoring, otherwise `.false.` |
| `restore_timescale` | `90` |
| `restore_mask` / `restore_width` / `restore_data` | `'constant'` / `1` / `'initial'` |
| `restore_flds` | `'aicen','vicen','vsnon','trcrn','velocity'` |
| `ice_ic` | the generated restoring file when restoring, otherwise `'default'` |
| `restart_aero`, `restart_age`, `restart_fsd`, `restart_fy`, `restart_lvl`, `restart_pond_sealvl`, `restart_snow` | `.true.` when restoring, otherwise `.false.` |

To start from a previous run's ice state, see the warm-start note in
[Case Setup, Section 3](case_setup.ipynb#case-cice).

(forcings-ww3)=
## Section 8: Waves (WW3)

The compset choice and the grid input files WW3 needs are covered in
[Case Setup, Section 5](case_setup.ipynb#case-ww3). Here's the `configure_forcings` half.

WW3's boundary forcing is a set of 2-D spectra, energy density `E(f, θ)`, at stations
along your open boundaries. It runs through the same GET → REGRID → MERGE engine as MOM6's
OBC, so the same date-chunking controls apply.


In [ ]:
case.configure_forcings(
    date_range=["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    # ... the usual MOM6 IC/OBC arguments from Sections 1-2 ...
    ww3_obc_product_name="era5_wave_spectra",   # default
    ww3_obc_function_name="get_era5_2d_spectra",  # default
    get_step_days=1,      # one CDS request per day per boundary
    regrid_step_days=1,
)

### The arguments

| Argument | Default | What it's for |
|---|---|---|
| `ww3_obc_product_name` | required | Which product to source boundary spectra from, or `"none"` to run without boundary spectra |
| `ww3_obc_function_name` | required (unless the product is `"none"`) | Which access function on that product to call |
| `ww3_obc_function_overrides` | `None` | Overrides for the access function's own defaulted arguments, e.g. `{"cdsapi_rc_path": "~/.cdsapirc_cds"}` |
| `get_step_days` | whole `date_range` in one request | Chunk the GET step. Smaller requests turn around faster on rate-limited APIs, and a resumed run skips any chunk already on disk |
| `regrid_step_days` | whole `date_range` | Same, for the REGRID step |

`boundaries` is shared with MOM6's OBC, the sides you already pass in Section 1 are the
sides WW3 generates spectra for.

### The products

| Product | Access function | Needs |
|---|---|---|
| `era5_wave_spectra` | `get_era5_2d_spectra` | ERA5's true 2-D spectrum (ECMWF param 251.140) from CDS's `reanalysis-era5-complete`, **not** the bulk wave stats on `reanalysis-era5-single-levels`. Requires its own CDS key |
| `CESM-WW3-JRA` | `get_cesm_ww3_jra_spectra` | Read access on GLADE to a global, JRA-forced CESM/WW3 run's restarts (defaults to `wi_jra.glo1p0.001`, useful from about 2018-12-01). No network or credentials, and its winds match a JRA-forced regional case |
| `reference_waves` | `get_reference_wave_spectra` | Nothing. Synthetic JONSWAP-shaped spectra with cosine-2s directional spreading, in the same shape the decoded ERA5 product uses. No network at all |

:::{note}
`era5_wave_spectra` needs a **separate** CDS API key from the one GLOFAS uses. Register at
[cds.climate.copernicus.eu](https://cds.climate.copernicus.eu), accept the
`reanalysis-era5-complete` licence, and write your credentials to `~/.cdsapirc_era5`:

```{code-block} text
url: https://cds.climate.copernicus.eu/api
key: <UID>:<API-key>
```

Point somewhere else with the `cdsapi_rc_path` argument via `ww3_obc_function_overrides`.
:::

### What it writes

`process_forcings` writes the per-station boundary spectrum files into the case's wave
input directory, along with the `spec.list` that lists them and the `ww3_bounc.nml` that
reads them (`INTERP = 2`, linear interpolation between stations). It also sets
`WW3_GRID_INP_DIR` in the case XML to point at that directory.

## Next steps

Once configured, move to [Process Forcings](process_forcings.ipynb) to actually
generate the files.